In [ ]:
import logging

from library.junction_patch import JunctionPatch
from library.qubit_array import QubitArray
from library.surface_code.expanding_patch import ExpandingSurfaceCodePatch
from library.surface_code.patch import SurfaceCodePatch, PauliBasis
from library.steane_code.patch import SteaneCodePatch

logging.basicConfig(level=logging.ERROR)
import stim
from IPython.display import Markdown
from utils.error_rate_analyser import simulate

In [ ]:
def detector_report(circuit: stim.Circuit) -> str:
    gauge_found = False
    try:
        circuit.detector_error_model(allow_gauge_detectors=False)
    except ValueError:
        gauge_found = True

    missing = len(circuit.missing_detectors())

    if gauge_found and missing:
        return "MISSING&GAUGE"
    elif missing:
        return "MISSING"
    elif gauge_found:
        return "GAUGE"

    return ""

In [ ]:
TARGET_DISTANCE = 9
SUPERDENSE_ROUNDS = 3
TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 9:
    raise ValueError("TARGET_DISTANCE must be odd and above 9.")

In [ ]:
# circuit = stim.Circuit()
# array = QubitArray(circuit, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
#
# steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7))
# junction = JunctionPatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-5))
# source = SurfaceCodePatch(array, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4))
# inactive_source = lambda location: location[1] == TARGET_DISTANCE - 4.5
# expanding = ExpandingSurfaceCodePatch(array, distance=5, anchor=(1, 1), expansion=TARGET_DISTANCE - 5)

scenarios: dict[str, stim.Circuit] = dict()
point = 0

In [ ]:
# Generate circuit up to and including preparation (w/ S-injection)
circuit = stim.Circuit()
array = QubitArray(circuit, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7))

# Append preparation
steane.append_preparation(circuit)
steane.append_observable(circuit, observable='Y', support = steane.logical)
scenarios[f'Point {point} - Prepared'] = circuit
point += 1

In [ ]:
# Generate circuit up to and including superdense code cycles
circuit = stim.Circuit()
array = QubitArray(circuit, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7))

steane.append_preparation(circuit)
for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuit, prefix=label)

steane.annotate_detectors(circuit, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)
steane.append_observable(circuit, observable='Y', support = steane.logical)

scenarios[f'Point {point} - SDCx{SUPERDENSE_ROUNDS}'] = circuit
point += 1

In [ ]:
# Generate circuit up to and including double-check-S
circuit = stim.Circuit()
array = QubitArray(circuit, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7))

steane.append_preparation(circuit)
for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuit, prefix=label)
steane.append_cultivation(circuit, prefix="CULT")

steane.annotate_detectors(circuit, sdc_rounds=SUPERDENSE_ROUNDS)
steane.append_observable(circuit, observable='Y', support = steane.logical)

scenarios[f'Point {point} - Double-Check-S'] = circuit
point += 1

In [ ]:
# Generate circuit up to and including teleportation
circuit = stim.Circuit()
array = QubitArray(circuit, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7))
junction = JunctionPatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-5))
source = SurfaceCodePatch(array, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4))
inactive_source = lambda location: location[1] == TARGET_DISTANCE - 4.5

steane.append_preparation(circuit)
for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuit, prefix=label)
steane.append_cultivation(circuit, prefix="CULT")
for rnd in range(TELEPORT_ROUNDS):
    for mmt in steane.TELEPORTATION_MOMENTS:
        steane.append_teleportation_slice(circuit, moment=mmt, prefix=f"TPT{rnd}")
        junction.append_syndrome_slice(circuit, moment=mmt, prefix=f"JCT{rnd}")
        source.append_round_slice(
            circuit, moment=mmt, prepare=PauliBasis.X if rnd == 0 else None, prefix=f"SC{rnd}",
            inactive = inactive_source
        )
        circuit.append("TICK")
steane.append_destruction(circuit)
source.append_round(circuit, prefix=f"SC{TELEPORT_ROUNDS}")

source.append_general_observable(circuit, {
    (0,0) : "Y", (1,0) : "Z", (2,0) : "Z", (3,0) : "Z", (4,0) : "Z", (0,1) : "X", (0,2) : "X", (0,3) : "X", (0,4) : "X",
}, "JCT0:Z0", "JCT0:Z1", "JCT0:Z2", "TPT0:XB", "TPT1:XB", "TPT2:XB", "DST:X1", "DST:X5", "DST:X6")

steane.annotate_detectors(circuit, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)
junction.annotate_detectors(circuit, rounds=TELEPORT_ROUNDS)
source.annotate_detectors(circuit, rounds=TELEPORT_ROUNDS+1, prepared=PauliBasis.X)

scenarios[f'Point {point} - Teleported'] = circuit
point += 1

In [ ]:
# Generate the circuit up to and including expansion :)
circuit = stim.Circuit()
array = QubitArray(circuit, dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
steane = SteaneCodePatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-7))
junction = JunctionPatch(array, anchor=(TARGET_DISTANCE-5, TARGET_DISTANCE-5))
source = SurfaceCodePatch(array, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4))
inactive_source = lambda location: location[1] == TARGET_DISTANCE - 4.5
expanding = ExpandingSurfaceCodePatch(array, distance=5, anchor=(1, 1), expansion=TARGET_DISTANCE - 5)

steane.append_preparation(circuit)
for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuit, prefix=label)
steane.append_cultivation(circuit, prefix="CULT")
for rnd in range(TELEPORT_ROUNDS):
    for mmt in steane.TELEPORTATION_MOMENTS:
        steane.append_teleportation_slice(circuit, moment=mmt, prefix=f"TPT{rnd}")
        junction.append_syndrome_slice(circuit, moment=mmt, prefix=f"JCT{rnd}")
        source.append_round_slice(
            circuit, moment=mmt, prepare=PauliBasis.X if rnd == 0 else None, prefix=f"SC{rnd}",
            inactive = inactive_source
        )
        circuit.append("TICK")
steane.append_destruction(circuit)
source.append_round(circuit, prefix=f"SC{TELEPORT_ROUNDS}")
for mmt in expanding.MOMENTS:
    expanding.append_expansion_slice(circuit, moment=mmt, prefix=f"EXP")
    circuit.append("TICK")

logical_observable = { (4,4) : "Y" }
for i in range(9):
    if i == 4:
        continue
    logical_observable[(i,4)] = "Z"
    logical_observable[(4,i)] = "X"
expanding.append_general_observable(
    circuit, logical_observable, "JCT0:Z0", "JCT0:Z1", "JCT0:Z2", "TPT0:XB", "TPT1:XB", "TPT2:XB", "DST:X1", "DST:X5", "DST:X6"
)

steane.annotate_detectors(circuit, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)
junction.annotate_detectors(circuit, rounds=TELEPORT_ROUNDS)
source.annotate_detectors(circuit, rounds=TELEPORT_ROUNDS+1, prepared=PauliBasis.X)
expanding.annotate_detectors(circuit, sc_rounds=TELEPORT_ROUNDS+1, source=source)

scenarios[f'Point {point} - Expanded'] = circuit
point += 1

In [ ]:
for point, circuit in scenarios.items():
    warning = detector_report(circuit)
    display(Markdown(f"[Open in Crumble ({point})]({circuit.to_crumble_url()}) {warning}"))

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Magic State Cultivation of $|\mathbf{S}\rangle$ [$\mathbf{Y}$ observable]"
simulate(scenarios, title, postselection=True, shots=1e7, minimal_noise=-7, figsize=(11, 4.5), num_workers=7)